# 単語の頻度だけでかな漢字変換をする (ビタビアルゴリズム)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_japanese_input/kkc_unigram.ipynb)

NLP クックブック 27-04 ／ 日本語入力 (かな漢字変換)

ひらがなの列を、漢字かな交じりの文に変換したい。最も単純な方法として、単語の出現頻度だけを使って、全体として最も確からしい単語の並びを選ぶ。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import io, math, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
import marisa_trie
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
sentences = []
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        if s.strip():
            sentences.append([(w.surface, jaconv.kata2hira(w.feature.kana or w.surface)) for w in tagger(s.strip())])

dic = defaultdict(Counter)                    # 読み → {表記: 回数}
for sent in sentences:
    for surface, reading in sent:
        dic[reading][surface] += 1
total = sum(sum(c.values()) for c in dic.values())
trie = marisa_trie.Trie(dic.keys())

In [3]:
UNKNOWN_COST = 20.0        # 辞書にない 1 文字をそのまま通すときのコスト

def lattice(kana):
    """各位置から始まる単語の候補 (終了位置, 表記, コスト) を列挙する。コスト = −log 確率"""
    edges = defaultdict(list)
    for i in range(len(kana)):
        for reading in trie.prefixes(kana[i:]):
            for surface, count in dic[reading].items():
                edges[i].append((i + len(reading), surface, -math.log(count / total)))
        edges[i].append((i + 1, kana[i], UNKNOWN_COST))      # 未知語: 1 文字をひらがなのまま出す
    return edges

def convert(kana):
    """ビタビアルゴリズム: 各位置までの最小コストを前から順に求め、最後から経路をたどる"""
    edges = lattice(kana)
    best = [(math.inf, None, None)] * (len(kana) + 1)        # (コスト, 直前の位置, 表記)
    best[0] = (0.0, None, None)
    for i in range(len(kana)):
        if best[i][0] == math.inf:
            continue
        for j, surface, cost in edges[i]:
            if best[i][0] + cost < best[j][0]:
                best[j] = (best[i][0] + cost, i, surface)
    words, pos = [], len(kana)
    while pos > 0:
        _, prev, surface = best[pos]
        words.append(surface); pos = prev
    return words[::-1], best[len(kana)][0]

for kana in ["わがはいはねこである", "なまえはまだない", "きょうはいいてんきだ", "がっこうへいく", "せんせいにてがみをかく", "はしのうえではしをおとした", "きしゃがきしゃできしゃした"]:
    words, cost = convert(kana)
    print(f"{kana} → {'|'.join(words)}  (コスト {cost:.1f})")

わがはいはねこである → 吾輩|は|猫|で|ある  (コスト 26.4)
なまえはまだない → 名前|浜田|ない  (コスト 23.9)
きょうはいいてんきだ → 今日|は|いい|天気|だ  (コスト 32.2)
がっこうへいく → 学校|へ|行く  (コスト 20.1)
せんせいにてがみをかく → 先生|に|手紙|を|かく  (コスト 29.7)
はしのうえではしをおとした → は|し|の|上|で|は|し|を|落し|た  (コスト 46.4)
きしゃがきしゃできしゃした → 汽車|が|汽車|で|汽車|し|た  (コスト 43.9)


In [4]:
# ラティスの中身を見る: 「はし」の位置にどんな候補があるか
edges = lattice("はしをわたる")
for i in [0, 3]:
    print(i, sorted([(s, round(c, 1), j) for j, s, c in edges[i]], key=lambda x: x[1])[:8])

0 [('は', 3.4, 1), ('ハ', 9.3, 1), ('箸', 9.6, 2), ('歯', 9.9, 1), ('端', 10.2, 2), ('葉', 10.3, 1), ('橋', 10.8, 2), ('派', 11.7, 1)]
3 [('わ', 8.1, 4), ('羽', 10.4, 4), ('輪', 11.0, 4), ('和', 11.0, 4), ('渡る', 11.2, 6), ('把', 11.7, 4), ('綿', 12.8, 5), ('わ', 20.0, 4)]


- 入力の各位置から始まる単語の候補を並べたグラフをラティスと呼ぶ。変換は、ラティスの左端から右端までの最もコストの低い経路を探す問題になる。
- 単語の頻度だけ (unigram) だと、同音異義語は文脈に関係なく、常に最も頻度の高い表記になる (「きしゃ」はすべて「汽車」)。
- 短い単語を細かくつなぐ経路と、長い単語 1 つの経路のどちらが選ばれるかは、コストの合計だけで決まる。頻度の高い助詞をつないだ「は|し」が「橋」や「箸」に勝ったり、「は|まだ」より「浜田」が選ばれたりするのは、単語の並びの自然さを見ていないため。
- コーパスが明治の小説なので、現代の語 (「天気」はあっても「スマホ」はない) や文体には弱い。辞書とコーパスが変換の質を決める。